# 01 — Snapshot Inventory (R2)

**Fecha:** 2026-02-02  
**Objetivo:** Crear un manifest reproducible (slice) y revisar inventario de objetos: conteos, datasets, particiones y ejemplos de keys.

**Slice actual:**
- OHLCV 1m: `ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/`
- Quotes: `quotes_p95/AABA/year=2019/month=01/`

**Outputs esperados:**
- `data/manifests/r2_slice_AABA_2019_01.json`
- Resumen de conteos y ejemplos


In [1]:
import os
import sys
import json
import subprocess
from datetime import datetime

print("python:", sys.executable)
print("cwd:", os.getcwd())
print("time:", datetime.utcnow().isoformat(), "UTC")

# Comprobar que podemos importar settings (clave: reproducibilidad)
from src.core.settings import settings

print("\n[settings]")
print("R2_BUCKET:", settings.R2_BUCKET)
print("R2_ENDPOINT:", settings.R2_ENDPOINT)
print("DATA_CACHE_DIR:", settings.DATA_CACHE_DIR)
print("RUNS_DIR:", settings.RUNS_DIR)


python: /Users/alex/Desktop/TSIS.AI/Backtesting_system/backtest/bin/python
cwd: /Users/alex/Desktop/TSIS.AI/Backtesting_system
time: 2026-02-02T22:32:30.834461 UTC


/var/folders/p5/jp6dx55n0tl16f0fwb0jj6t00000gn/T/ipykernel_60659/2150899082.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print("time:", datetime.utcnow().isoformat(), "UTC")



[settings]
R2_BUCKET: tsis-data
R2_ENDPOINT: https://1c485fe75c8be34cf211021abedc8105.r2.cloudflarestorage.com
DATA_CACHE_DIR: ./data/cache
RUNS_DIR: ./runs


## Definición del slice y ruta del manifest

Usaremos un slice pequeño (AABA, 2019-01) para validar el pipeline completo sin listar millones de objetos.


In [2]:
PREFIX_OHLCV = "ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/"
PREFIX_QUOTES = "quotes_p95/AABA/year=2019/month=01/"
MANIFEST_PATH = "data/manifests/r2_slice_AABA_2019_01.json"

print(PREFIX_OHLCV)
print(PREFIX_QUOTES)
print("manifest:", MANIFEST_PATH)


ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/
quotes_p95/AABA/year=2019/month=01/
manifest: data/manifests/r2_slice_AABA_2019_01.json


In [3]:
cmd = [
    sys.executable, "-u", "-m", "scripts.build_manifest",
    "--prefix", PREFIX_OHLCV,
    "--prefix", PREFIX_QUOTES,
    "--out", MANIFEST_PATH
]

print("Running:\n", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("\n[stdout]\n", result.stdout)
print("\n[stderr]\n", result.stderr)
print("returncode:", result.returncode)

assert result.returncode == 0, "build_manifest failed"


Running:
 /Users/alex/Desktop/TSIS.AI/Backtesting_system/backtest/bin/python -u -m scripts.build_manifest --prefix ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/ --prefix quotes_p95/AABA/year=2019/month=01/ --out data/manifests/r2_slice_AABA_2019_01.json

[stdout]
 ✅ Manifest written: data/manifests/r2_slice_AABA_2019_01.json
{
  "listed_objects_total": 22,
  "recognized_objects": 22,
  "unrecognized_objects": 0
}


[stderr]
 
returncode: 0


**Code (Cargar el manifest y ver conteos)**

In [4]:
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest["counts"]


{'listed_objects_total': 22,
 'recognized_objects': 22,
 'unrecognized_objects': 0}

In [9]:
objs = manifest["objects"]
len(objs), objs[0].keys()


(22,
 dict_keys(['dataset', 'day', 'era', 'etag', 'key', 'last_modified', 'month', 'size', 'symbol', 'year']))

In [10]:
# Mostrar algunas keys de ejemplo
for o in objs[:10]:
    print(o["dataset"], "|", o["key"])


ohlcv_intraday_1m | ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/minute.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=03/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=04/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=07/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=08/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=09/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=10/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=11/quotes.parquet
quotes_p95 | quotes_p95/AABA/year=2019/month=01/day=14/quotes.parquet


**Code (Tabla y agregados por dataset/particiones)**

In [13]:
# Code (Inspección rápida: primeras keys)
import pandas as pd

df = pd.DataFrame(objs)
df.head(1).T


,0
dataset,ohlcv_intraday_1m
day,NaN
era,2019_2025
etag,86d58162950b13f5d8c9496223b6b414
key,ohlcv_intraday_1m/2019_2025/AABA/year=2019/mon...
last_modified,2026-01-20T09:11:38.643000+00:00
month,1
size,188753
symbol,AABA
year,2019


In [ ]:
df["dataset"].value_counts()

dataset
quotes_p95           21
ohlcv_intraday_1m     1
Name: count, dtype: int64

In [15]:
df_quotes = df[df["dataset"].isin(["quotes_p95", "quotes_test"])].copy()
df_quotes[["year", "month", "day"]].value_counts().sort_index().head(20)

year  month  day 
2019  1      2.0     1
             3.0     1
             4.0     1
             7.0     1
             8.0     1
             9.0     1
             10.0    1
             11.0    1
             14.0    1
             15.0    1
             16.0    1
             17.0    1
             18.0    1
             22.0    1
             23.0    1
             24.0    1
             25.0    1
             28.0    1
             29.0    1
             30.0    1
Name: count, dtype: int64

## Conclusión

- El parser reconoce el 100% de keys del slice (`unrecognized_objects=0`).
- El manifest generado define exactamente qué objetos se usarán en el siguiente paso (sync + validation).
- Siguiente paso: descargar SOLO estos objetos en `data/cache/` con `scripts.r2_sync` y validar con `scripts.validate_data`.


In [16]:
SYNC_CMD = f"""
{sys.executable} -u -m scripts.r2_sync \
  --manifest {MANIFEST_PATH} \
  --max_workers 4 \
  --limit 2
""".strip()

VALIDATE_CMD = f"""
{sys.executable} -m scripts.validate_data \
  --manifest {MANIFEST_PATH} \
  --out_dir runs/data_quality/AABA_2019_01
""".strip()

print("NEXT (sync test 2 files):\n", SYNC_CMD)
print("\nNEXT (validate after sync):\n", VALIDATE_CMD)


NEXT (sync test 2 files):
 /Users/alex/Desktop/TSIS.AI/Backtesting_system/backtest/bin/python -u -m scripts.r2_sync   --manifest data/manifests/r2_slice_AABA_2019_01.json   --max_workers 4   --limit 2

NEXT (validate after sync):
 /Users/alex/Desktop/TSIS.AI/Backtesting_system/backtest/bin/python -m scripts.validate_data   --manifest data/manifests/r2_slice_AABA_2019_01.json   --out_dir runs/data_quality/AABA_2019_01


In [17]:
import json

metrics_path = "runs/data_quality/AABA_2019_01/quality_metrics.json"
anom_path = "runs/data_quality/AABA_2019_01/anomalies.jsonl"

with open(metrics_path, "r", encoding="utf-8") as f:
    m = json.load(f)

print("OVERALL:", m["overall"])
print("FAIL:", m["fail"], "WARN:", m["warn"], "INFO:", m["info"])

print("\nFirst 5 anomalies:")
with open(anom_path, "r", encoding="utf-8") as f:
    for i, line in zip(range(5), f):
        a = json.loads(line)
        print(a["severity"], a["message"])
        print("  ", a["key"])


OVERALL: FAIL
FAIL: 21 WARN: 0 INFO: 0

First 5 anomalies:
FAIL Missing required columns
   quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet
FAIL Missing required columns
   quotes_p95/AABA/year=2019/month=01/day=03/quotes.parquet
FAIL Missing required columns
   quotes_p95/AABA/year=2019/month=01/day=04/quotes.parquet
FAIL Missing required columns
   quotes_p95/AABA/year=2019/month=01/day=07/quotes.parquet
FAIL Missing required columns
   quotes_p95/AABA/year=2019/month=01/day=08/quotes.parquet


**Data Integrity Gate: resultado y anomalías**

In [18]:
# --- Data Integrity Gate: Result Summary (Diary Capture) ---

import json
from collections import Counter
from pathlib import Path

RUN_DIR = Path("runs/data_quality/AABA_2019_01")
METRICS_PATH = RUN_DIR / "quality_metrics.json"
ANOMALIES_PATH = RUN_DIR / "anomalies.jsonl"

assert METRICS_PATH.exists(), f"Missing {METRICS_PATH}"
assert ANOMALIES_PATH.exists(), f"Missing {ANOMALIES_PATH}"

# Load metrics
with METRICS_PATH.open("r", encoding="utf-8") as f:
    metrics = json.load(f)

print("=== DATA INTEGRITY RESULT ===")
print("Overall status:", metrics["overall"])
print(
    "Files checked:",
    metrics["checked_files"],
    "| OHLCV:",
    metrics["ohlcv_files"],
    "| Quotes:",
    metrics["quotes_files"],
)
print("Anomalies → FAIL:", metrics["fail"], "WARN:", metrics["warn"], "INFO:", metrics["info"])

# Load anomalies
anomalies = []
with ANOMALIES_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        anomalies.append(json.loads(line))

# Aggregate anomalies by (severity, message)
counter = Counter((a["severity"], a["message"]) for a in anomalies)

print("\n=== TOP ANOMALIES ===")
for (sev, msg), n in counter.most_common(10):
    print(f"{sev:4} {n:5}  {msg}")

# Show a few concrete examples for traceability
print("\n=== SAMPLE ANOMALIES (first 5) ===")
for a in anomalies[:5]:
    print(f"[{a['severity']}] {a['message']}")
    print("  key:", a["key"])
    if "details" in a and a["details"]:
        print("  details:", a["details"])


=== DATA INTEGRITY RESULT ===
Overall status: WARN
Files checked: 22 | OHLCV: 1 | Quotes: 21
Anomalies → FAIL: 0 WARN: 15 INFO: 0

=== TOP ANOMALIES ===
WARN    15  Crossed market (bid > ask) rows

=== SAMPLE ANOMALIES (first 5) ===
[WARN] Crossed market (bid > ask) rows
  key: quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet
  details: {'rows': 6}
[WARN] Crossed market (bid > ask) rows
  key: quotes_p95/AABA/year=2019/month=01/day=08/quotes.parquet
  details: {'rows': 11}
[WARN] Crossed market (bid > ask) rows
  key: quotes_p95/AABA/year=2019/month=01/day=09/quotes.parquet
  details: {'rows': 3}
[WARN] Crossed market (bid > ask) rows
  key: quotes_p95/AABA/year=2019/month=01/day=11/quotes.parquet
  details: {'rows': 1}
[WARN] Crossed market (bid > ask) rows
  key: quotes_p95/AABA/year=2019/month=01/day=14/quotes.parquet
  details: {'rows': 9}



### ✅ “Validation complete: WARN” + “Crossed market (bid > ask) rows”

En algunos registros de tu dataset de quotes aparece **bid_price > ask_price** (el mejor precio comprador es mayor que el mejor vendedor). Eso se llama **mercado cruzado** (*crossed market*). En teoría “no debería pasar”, porque el bid debería ser ≤ ask.

Pero en datos reales sí aparece por varios motivos legítimos:

* **desincronización temporal** entre fuentes/venues (latencia, consolidación)
* **microstructure noise** (updates casi simultáneos)
* **datos consolidados** con timestamping imperfecto (por ejemplo, `participant_timestamp` vs `timestamp`)
* **errores puntuales** o condiciones de mercado especiales

Por eso lo marcamos como **WARN**: no es necesariamente “corrupción”, pero **sí afecta a simulación de fills** si no lo modelas.

**Implicación práctica:**
Para backtesting, cuando hay bid>ask, debes decidir una regla:

* (A) “Clamp”: forzar `bid = min(bid, ask)` o `ask = max(ask, bid)`
* (B) “Drop”: descartar esos ticks/filas
* (C) “Swap”: intercambiar bid/ask si viene invertido (poco recomendable sin evidencia)

En el 2º notebook vamos a **cuantificar cuánto ocurre** (ratio, por día) y decidir la política.
